# 6 Information theory

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

## 6.1 Surprisal

First, let's set up a probability space and base measure. Notice that we are introducing both $P$- and $\mu$-null atoms in our definitions.

In [2]:
import sigalg as sa

product_space = sa.SampleSpace.from_sequence(size=7)
F = sa.SigmaAlgebra(
    domain=product_space,
    mapping={
        0: 0,
        1: 1,
        2: 2,
        3: 2,
        4: 3,
        5: 4,
        6: 4,
    },
)
P = sa.ProbabilityMeasure(
    domain=F,
    mapping={
        0: 0.2,
        1: 0.6,
        2: 0.2,
        3: 0.0,  # null atom
        4: 0.0,  # null atom
    },
)
mu = sa.Measure(
    domain=F,
    mapping={
        0: 1,
        1: 2,
        2: 3,
        3: 4,
        4: 0,  # null atom
    },
)

Since atom `4` is `mu`-null, in order to maintain absolute continuity, it must be `P`-null as well. But `P` has an extra null atom (with identifier `3`) that is not `mu`-null.

We compute the surprisal $s(P;\mu)$ through the `surprisal` method of the probability measure `P`. It returns an `F`-measurable instance of `RandomVariable`.

In [3]:
surprisal = P.surprisal(mu)
surprisal

RandomVariable(parameters=(omega), domain=Omega, sig_alg=F, measure=P, name=s(P; mu))

In [4]:
print(surprisal)

Random variable 's(P; mu)':
       s(P; mu)
omega          
0      1.609438
1      1.203973
2      2.708050
3      2.708050
4      0.000000
5      0.000000
6      0.000000


Note that the surprisal is `0` on the null atoms.

The `surprisal` accepts an optional parameter `base`, allowing the user to select the base of the logarithm that is called. The default base is `'e'` for the natural logarithm — other choices for `base` are `'2'` and `'10'`.

By definition, the surprisal is the negative logarithm of the Radon-Nikodym of the probability measure with respect to the base measure. We can check this directly in SigAlg.

In [5]:
import numpy as np

with np.errstate(divide="ignore"):  # Necessary to avoid a divide-by-0 exception
    result = -np.log(P.derivative(mu))

print(result)

Measurable function '(-log(dP_dmu))':
       (-log(dP_dmu))
omega                
0            1.609438
1            1.203973
2            2.708050
3            2.708050
4                 inf
5                 inf
6                 inf


Notice the `inf` values on the null atoms, which the `surprisal` method automatically replaces with `0`.

## 6.2 Entropy

We obtain the entropy of the measure `P` by calling the `entropy` method, being sure to pass in the base measure.

In [6]:
P.entropy(mu)

1.5858813053028238

By definition, the entropy is the expected value of the surprisal:

$$
H(P;\mu) = \int_\Omega s(P;\mu) \, dP.
$$

We may chain the `surprisal` and `integrate` methods to check that our computation of the entropy is correct.

In [7]:
P.surprisal(mu).integrate()

1.5858813053028238

## 6.3 Kullback Leibler divergence and mutual information

To demonstrate the KL divergence, we introduce a second probability measure.

In [8]:
Q = sa.ProbabilityMeasure(
    domain=F,
    mapping={
        0: 0.35,
        1: 0.4,
        2: 0.25,
        3: 0.0,  # null atom
        4: 0.0,  # null atom
    },
)

P.divergence(Q)

0.08672719701497207

As we mentioned in the mathematical reference, the KL divergence $D(P \parallel Q)$ is not generally symmetric in $P$ and $Q$. Our current measures provide an example of this asymmetry.

In [9]:
Q.divergence(P)

0.08946537036268461

In [10]:
Omega = sa.SampleSpace.from_sequence(size=2)
print(Omega ^ 2)

Sample space 'Omega ^ 2':
 omega_0  omega_1
       0        0
       0        1
       1        0
       1        1


In [11]:
P = sa.ProbabilityMeasure(
    domain=Omega ^ 2,
    mapping={
        (0, 0): 0.12,
        (0, 1): 0.18,
        (1, 0): 0.28,
        (1, 1): 0.42,
    },
)
print(P)

Probability measure 'P':
                    P
omega_0 omega_1      
0       0        0.12
        1        0.18
1       0        0.28
        1        0.42


In [12]:
P1 = sa.ProbabilityMeasure(
    domain=Omega, mapping=dict(zip(Omega, [0.3, 0.7])), name="P1"
)
P2 = sa.ProbabilityMeasure(
    domain=Omega, mapping=dict(zip(Omega, [0.4, 0.6])), name="P2"
)

print(sa.ProbabilityMeasure.tensor_product([P1, P2]))

Probability measure 'P1 x P2':
                 P1 x P2
omega_0 omega_1         
0       0           0.12
        1           0.18
1       0           0.28
        1           0.42


In [13]:
S = sa.SampleSpace.from_sequence(size=3, variable_name="s", name="S")
F = sa.SigmaAlgebra(domain=S, mapping=dict(zip(S, [0, 1, 1])))
T = sa.SampleSpace.from_sequence(size=3, variable_name="t", name="T")
G = sa.SigmaAlgebra(domain=T, mapping=dict(zip(T, [0, 0, 1])), name="G")
P = sa.ProbabilityMeasure.from_rand(domain=F @ G, random_state=42)
print(P)

Probability measure 'P':
            P
F G          
0 0  0.476401
  1  0.271161
1 0  0.000059
  1  0.252379


In [14]:
P.mutual_info(["F"], ["G"])

np.float64(-0.20185366179714626)

In [15]:
P_F = P.marginal(["F"])
P_G = P.marginal(["G"])

In [ ]:
# sa.ProbabilityMeasure.tensor_product([P_F, P_G])

TypeError: All elements of `factors` must be instances of Function.